# 04주 · 참오차·근사오차·부동소수점

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

**수업**: 2027-03-23 반올림 오차 (1) · 대면 / 2027-03-25 반올림 오차 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week04/CE273_W04.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week04)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def series_table(kind, x, terms=10):
    if kind not in ["exp","sin","geometric"] or not np.isfinite(x) or not isinstance(terms,int) or not 1<=terms<=100:
        raise ValueError("종류·유한한 x·1~100의 정수 항 수 확인 필요")
    rows=[]; total=0.; previous=None
    truth={"exp":lambda:math.exp(x),"sin":lambda:math.sin(x),"geometric":lambda:1/(1-x)}[kind]()
    for k in range(terms):
        term = x**k/math.factorial(k) if kind=="exp" else ((-1)**k*x**(2*k+1)/math.factorial(2*k+1) if kind=="sin" else x**k)
        total += term
        step=np.nan if previous is None or total==0 else abs((total-previous)/total)*100
        rows.append({"k":k,"term":term,"estimate":total,"absolute_error":abs(truth-total),"true_relative_percent":100*relative_error(truth,total),"step_percent":step})
        previous=total
    return pd.DataFrame(rows)


## S01. e^0.5의 급수와 종료 기준

**슬라이드 연결**: 2026_S1_NA_Wk4a_Roundoff1.pptx p10, 12, 13, 14, 15, 16 / 2026_S1_NA_Wk4b_Roundoff2.pptx p3, 6

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 참오차와 이전 반복과의 차이는 같은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
table=series_table("exp",.5,10);threshold=.5*10**(2-3)
print(table)
first=table.loc[table.step_percent<threshold].iloc[0]
print("처음 만족:",first,"허용오차(%):",threshold)
assert int(first.k)==5 and np.isclose(first.estimate,1.6486979166666667)
assert np.isclose(relative_error(20,19.5)*100,2.5)

**조건 변경**: 유효숫자 목표와 허용오차를 바꾸고 필요한 항 수 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: e^.5·n3 유효숫자·허용오차 .05%·최대 항 수·참값 별도 대조 요청

**결과 검토**: 원본 결과 표기의 반올림과 실제 합 구분. 작은 갱신량만으로 참오차 보장 불가

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. 머신 엡실론·상쇄와 안정식

**슬라이드 연결**: 2026_S1_NA_Wk4b_Roundoff2.pptx p12, 17, 18, 22, 23, 29

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: epsilon은 모든 크기에서의 최소 절대 간격인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
info=np.finfo(np.float64);gap=np.nextafter(1.,2.)-1
print("float64 epsilon:",info.eps,"gap at 1:",gap,"gap near 1e8:",np.spacing(1e8))
assert gap==2.0**-52 and np.spacing(1e8)>gap
from decimal import Decimal,localcontext
with localcontext() as context:
    context.prec=60
    a,b,c=map(Decimal,["1","3000.001","1"])
    reference=(-2*c)/(b+(b*b-4*a*c).sqrt())
a,b,c=np.float32(1),np.float32(3000.001),np.float32(1)
two,four=np.float32(2),np.float32(4)
disc=np.sqrt(b*b-four*a*c);naive=(-b+disc)/(two*a);stable=-two*c/(b+disc)
assert np.asarray(naive).dtype==np.float32
print({"naive_float32":naive,"stable_float32":stable,"decimal_reference":float(reference)})
print("0 근처의 최소 양수와 epsilon은 별개:",np.nextafter(0.,1.))

**조건 변경**: float32→float64와 큰 b를 바꿔 상쇄·안정식 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: dtype·식·스케일·반올림 방식·독립 고정밀 기준·오차 단위 명시

**결과 검토**: Python int의 가변 정밀도, float64의 53비트 유효 정밀도, CPU/SIMD 폭 구분

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

참오차·갱신량·표시 정밀도·연산 정밀도 구분

입력 항목:

- `e0_5_종료시_항수`: 해당 계산값·항 수·오차 기준의 수치
- `허용오차_퍼센트`: 해당 계산값·항 수·오차 기준의 수치
- `float64_epsilon`: 해당 계산값·항 수·오차 기준의 수치
- `epsilon은_전체_수직선의_최소_절대간격`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'e0_5_종료시_항수': None, '허용오차_퍼센트': None, 'float64_epsilon': None, 'epsilon은_전체_수직선의_최소_절대간격': None}


In [ ]:
def check_answers(values):
    expected = {'e0_5_종료시_항수': 6, '허용오차_퍼센트': 0.05, 'float64_epsilon': 2.220446049250313e-16, 'epsilon은_전체_수직선의_최소_절대간격': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
